# 04 — Scenario: Product Fit Problem

**The situation:** the Chelsea Parka's return rate has climbed materially,
concentrated in sizes M and L — a sizing/fit issue, not a general quality problem.

This notebook proves: returns anomaly detection, review/return-reason analysis,
financial impact, and fit diagnosis.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Investigate — return rate by size over time

In [2]:
style_id = con.execute("SELECT style_id FROM silver.dim_style WHERE style_name='Chelsea Parka'").fetchone()[0]
rr = con.execute(f"""
    SELECT week_start_date, size, return_rate FROM gold.returns_scorecard
    WHERE style_id = '{style_id}' AND size IN ('M','L') ORDER BY week_start_date
""").df()
fig = px.line(rr, x="week_start_date", y="return_rate", color="size",
              labels={"week_start_date": "", "return_rate": "Return rate", "size": "Size"})
fig.update_yaxes(tickformat=".0%")
style_fig(fig, "Chelsea Parka return rate by size — a clear step up in M and L")

## Investigate — reason codes and review text

In [3]:
reasons = con.execute(f"""
    SELECT reason_code, COUNT(*) n FROM silver.fact_returns_line
    WHERE style_id = '{style_id}' GROUP BY 1 ORDER BY 2 DESC
""").df()
fig = px.bar(reasons, x="reason_code", y="n", color_discrete_sequence=[CATEGORICAL[7]],
             labels={"reason_code": "", "n": "Return lines"})
style_fig(fig, "Chelsea Parka return reasons — fit dominates")

In [4]:
con.execute(f"""
    SELECT review_text FROM silver.fact_returns_line
    WHERE style_id = '{style_id}' AND review_text IS NOT NULL LIMIT 8
""").df()

,review_text
0,"Fit was tighter than expected for a Parka, ord..."
1,"Beautiful coat but too snug through the chest,..."
2,"Runs small in the shoulders, had to size up."
3,"Runs small in the shoulders, had to size up."
4,"Beautiful coat but too snug through the chest,..."
5,Loved the design but it ran small -- returning...
6,Loved the design but it ran small -- returning...
7,"Fit was tighter than expected for a Parka, ord..."


## Simulate — financial impact

In [5]:
impact = con.execute(f"""
    SELECT SUM(r.units_returned) units_returned, AVG(sku.current_retail_price) avg_price
    FROM silver.fact_returns_line r JOIN silver.dim_sku sku ON sku.sku_id = r.sku_id
    WHERE r.style_id = '{style_id}' AND sku.size IN ('M','L') AND r.reason_code = 'Fit-Small'
""").df().iloc[0]
processing_cost_per_unit = 18
total_cost = impact["units_returned"] * (impact["avg_price"] * 0.5 + processing_cost_per_unit)
print(f"Fit-driven M/L returns: {impact['units_returned']:.0f} units")
print(f"Estimated cost of returns to date: ${total_cost:,.0f} (restocking loss + processing)")

Fit-driven M/L returns: 7685 units
Estimated cost of returns to date: $7,209,810 (restocking loss + processing)


## Recommend

- Update the product page / size chart for M and L with a "runs small" note, and
  route new orders in these sizes toward a size-up prompt at checkout.
- Flag the silhouette to the design/buying team for the next seasonal drop.
- No broad quality or supplier issue — the signal is isolated to two sizes of one
  style, so this is a sizing fix, not a recall.